# 01 · Data collection and bilingual alignment

**Programming for AI · Topic 26 · Participant 1 · Nulimu Arey · AAI-2606M**

One row represents one review paired across English and Kazakh. This notebook verifies source provenance, checks the pairing and exports a clean bilingual corpus.

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "prepare_data.py").is_file():
    raise FileNotFoundError("Open this notebook from the project root or the notebooks directory.")
sys.path.insert(0, str(root))

## 1. Load the pinned source releases

The local CSVs are checked against the expected SHA-256 hashes. Missing files are downloaded from their pinned Hugging Face revisions. Each source file must contain 415 unique source row IDs.

In [2]:
from IPython.display import display
import pandas as pd
from src.data_collection import SOURCES, PROCESSED, load_sources, pair_reviews, source_url, write_csv

sources = load_sources()
manifest = pd.DataFrame([
    {"language": lang, "rows": len(frame), "revision": SOURCES[lang][1],
     "sha256": SOURCES[lang][3], "source": source_url(lang)}
    for lang, frame in sources.items()
])
display(manifest)

,language,rows,revision,sha256,source
0,en,415,bef7222ed5e05cb84bed63e31c28a77c40d389a6,4ccc35366b6cf6174221493c2f2070656faaa666a1b01f...,https://huggingface.co/datasets/Arailym-tleuba...
1,kk,415,d21d147364c2a35c9eba2a3791da793c9e06d95e,84f19e4926046bb24d69aa3bfc46edefb87cdd8c5f0882...,https://huggingface.co/datasets/Arailym-tleuba...


## 2. Pair and validate the reviews

Join on the published row index and require one row in each language. Validate the mapped labels, score missingness, citation counts and the one-to-one relationship between the normalized titles. Preserve the original bilingual model text apart from leading and trailing whitespace.

Map Төмен to Low, Орташа to Moderate and Жоғары to High. Keep unlabeled rows for corpus analysis. Remove comments, keywords and citations from the processed file.

In [3]:
reviews = pair_reviews(sources)
summary = pd.Series({
    "Review pairs": len(reviews),
    "Unique papers": reviews["paper_id"].nunique(),
    "Labeled pairs": reviews["label"].notna().sum(),
    "Unlabeled pairs": reviews["label"].isna().sum(),
}, name="Count")
display(summary.to_frame())
display(reviews[["review_id", "paper_id", "label"]].head(8))

,Count
Review pairs,415
Unique papers,215
Labeled pairs,385
Unlabeled pairs,30


,review_id,paper_id,label
0,0,p_3faf61b5e948e454,Moderate
1,1,p_3faf61b5e948e454,Moderate
2,2,p_cc34446db01da043,NaN
3,3,p_464fa19e2ada412a,Moderate
4,4,p_464fa19e2ada412a,Moderate
5,5,p_fb8a646fbe80d8ff,Moderate
6,6,p_fb8a646fbe80d8ff,Moderate
7,7,p_0e1b63de55cbb7ae,Moderate


## 3. Export the paired corpus

The model team receives separate language columns and a shared `review_id`. The paper identity is derived from the normalized English title and is shared by both language versions.

In [4]:
write_csv(reviews, PROCESSED)
print(f"Wrote {PROCESSED.relative_to(root)} with {len(reviews)} rows.")

Wrote data\processed\merged_dataset.csv with 415 rows.
